# Align a Language Model with DPO

Teach a small language model to prefer polite replies, then find out what beta really controls

Turn preference pairs into a loss and align a small language model without a reward model. Score replies by their log-probability, write the DPO loss and its implicit rewards, train a rude help desk model into a polite one, and measure how far it drifts when beta is too small.

## Load the base model

Loads PyTorch and the helpers for this notebook, then the base model, a 2-layer transformer language model with 114,304 learned weights that knows 95 words. It was trained before this notebook, first on short neutral sentences and then on chats from a made-up help desk where about half the replies are rude. That second step is supervised fine-tuning (SFT), and DPO always starts from such a model.

In [ ]:
import copy
import math
import torch
import torch.nn.functional as F
from dpo_lab import (VOCAB, load_base_model, count_parameters, make_preference_pairs, encode_pairs,
                     encode_chat, encode_sentences, neutral_sentences, test_prompts, generate,
                     evaluate_replies, perplexity, train_dpo, show_token_logprobs, bar)

base = load_base_model()
print("Vocabulary:", len(VOCAB), "words")
print("Parameters:", count_parameters(base))

## Meet the base model

The base model answers 6 requests whose verb and object it never saw together in training. Every reply carries out the request ("i will wash the table now .") and then ends politely or rudely. The bars count 120 sampled replies, which are about 0.54 polite, 0.46 rude, and 1.00 on task, meaning the reply repeats exactly the requested verb and object. Perplexity measures how surprised a model is by text, and about 3.4 here means that on neutral sentences it is as unsure as choosing among roughly 3 equally likely words at each step. You will watch this number to check that alignment does not damage general text.

In [ ]:
prompts = test_prompts()
for prompt, reply in zip(prompts[:6], generate(base, prompts[:6], seed=1)):
    print(f"{prompt:32s} -> {reply}")

base_stats, _ = evaluate_replies(base)
neutral = neutral_sentences()
base_perplexity = perplexity(base, neutral)
print(f"\nOn {len(prompts)} requests it never saw in training:")
print(f"  polite  {bar(base_stats['polite'])} {base_stats['polite']:.2f}")
print(f"  rude    {bar(base_stats['rude'])} {base_stats['rude']:.2f}")
print(f"  on task {bar(base_stats['on_task'])} {base_stats['on_task']:.2f}")
print(f"Perplexity on {len(neutral)} neutral sentences: {base_perplexity:.2f}")

## Look at the preference pairs

A preference pair is a prompt with two replies, the chosen one a person preferred and the rejected one. Here every chosen reply is polite and every rejected reply is rude, and both carry out the same request, so the only difference is the ending. This is exactly the data DPO trains on, with no scores and only a record of which reply won.

In [ ]:
pairs = make_preference_pairs()
print(len(pairs), "preference pairs. The first three:\n")
for pair in pairs[:3]:
    print("prompt:  ", pair["prompt"])
    print("chosen:  ", pair["chosen"])
    print("rejected:", pair["rejected"])
    print()

## Turn a chat into tokens and a mask

Each chat becomes one row of token ids. <bos> marks the start, then the prompt, <sep> separates prompt from reply, and <eos> ends the reply. The mask is 0 on the prompt and 1 on the reply, so only the reply is scored. Shorter chats are padded with <pad> and mask 0 so a batch can be one tensor.

In [ ]:
ids, mask = encode_chat(pairs[0]["prompt"], pairs[0]["chosen"])
print("position  token       mask")
for position, (token, m) in enumerate(zip(ids, mask)):
    print(f"{position:8d}  {VOCAB[token]:10s}  {m}")

## Score a reply with its log-probability

In [ ]:
def sequence_logprob(logits, tokens, mask):
    log_soft=F.log_softmax(logits,dim=-1)
    num_seqs=logits.shape[0]
    num_tokens=logits.shape[1]
    num_v=logits.shape[2]
    probs=torch.Tensor(num_seqs)
    for i in range(num_seqs):
        ans=0
        for j in range(num_tokens-1):
           ans+=log_soft[i][j][tokens[i][j+1]] *mask[i][j+1]
        probs[i]=ans
    return probs 
toy_logits = torch.zeros(1, 3, 4)
toy_logits[0, 0, 2] = math.log(3.0)
toy = sequence_logprob(toy_logits, torch.tensor([[0, 2, 1]]), torch.tensor([[0, 1, 1]]))
assert toy is not None and toy.shape == (1,), "Return one number per sequence, shape (n,)"
assert abs(toy.item() - (math.log(0.5) + math.log(0.25))) < 1e-5, "Token t is scored by the logits at position t - 1, and only where mask is 1"
assert abs(sequence_logprob(toy_logits, torch.tensor([[0, 2, 1]]), torch.tensor([[0, 0, 1]])).item() - math.log(0.25)) < 1e-5, "Skip tokens whose mask is 0"

data = encode_pairs(pairs)
with torch.no_grad():
    chosen_lp = sequence_logprob(base(data["chosen"][:4]), data["chosen"][:4], data["chosen_mask"][:4])
    rejected_lp = sequence_logprob(base(data["rejected"][:4]), data["rejected"][:4], data["rejected_mask"][:4])
print("chosen    rejected   pair")
for c, r, pair in zip(chosen_lp, rejected_lp, pairs):
    print(f"{c.item():7.2f}   {r.item():7.2f}    {pair['prompt']}")

## From two scores to a preference

In [ ]:
def bradley_terry(score_a, score_b):
        pref=score_a-score_b
        pref=1/(1+torch.exp(-pref))
        return pref
    

assert abs(bradley_terry(torch.tensor(2.0), torch.tensor(0.0)).item() - 0.8808) < 1e-4, "P(a beats b) = sigmoid(score_a - score_b)"
assert bradley_terry(torch.tensor(1.0), torch.tensor(1.0)).item() == 0.5, "Equal scores give 0.5"
assert bradley_terry(torch.tensor(1000.0), torch.tensor(0.0)).item() == 1.0, "Use torch.sigmoid so huge gaps stay finite"

with torch.no_grad():
    all_chosen = sequence_logprob(base(data["chosen"]), data["chosen"], data["chosen_mask"])
    all_rejected = sequence_logprob(base(data["rejected"]), data["rejected"], data["rejected_mask"])
p = bradley_terry(all_chosen, all_rejected)
print(f"Scoring each reply by the base model's own log-probability, over {len(p)} pairs:")
print(f"  average P(chosen beats rejected) = {p.mean().item():.3f}")

## Freeze a reference copy

DPO needs two models. The reference is a frozen copy of the base model, and requires_grad_(False) means it never changes, which the last line confirms with 0 trainable parameters. The policy is a second copy that will be trained (alignment borrows the word policy from reinforcement learning, where it means the model that chooses what to say). Because the reference never changes, its log-probabilities for all 600 pairs are computed once and stored, which saves a forward pass at every step. Right now both give identical numbers.

In [ ]:
reference = copy.deepcopy(base)
reference.requires_grad_(False)
policy = copy.deepcopy(base)

data["ref_chosen"], data["ref_rejected"] = all_chosen, all_rejected  # computed once, never again
with torch.no_grad():
    start = sequence_logprob(policy(data["chosen"][:4]), data["chosen"][:4], data["chosen_mask"][:4])
print("policy    reference   (log-probability of the first 4 chosen replies)")
for a, b in zip(start, data["ref_chosen"][:4]):
    print(f"{a.item():7.3f}   {b.item():7.3f}")
print("Reference parameters that can still change:", sum(p.requires_grad for p in reference.parameters()))

## Write the DPO loss

In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta):
    chosen_rewards=beta*(policy_chosen-ref_chosen)
    rejected_rewards=beta*(policy_rejected-ref_rejected)
    z=chosen_rewards-rejected_rewards
    loss=-F.logsigmoid(z).mean()
    return (loss,chosen_rewards,rejected_rewards)

same = torch.tensor([-3.0, -5.0])
loss, cr, rr = dpo_loss(same, same, same, same, 1.0)
assert abs(loss.item() - math.log(2)) < 1e-6, "When the policy equals the reference, every reward is 0 and the loss is log 2"
loss, cr, rr = dpo_loss(torch.tensor([-2.0]), torch.tensor([-6.0]), torch.tensor([-3.0]), torch.tensor([-3.0]), 0.5)
assert torch.allclose(cr, torch.tensor([0.5])) and torch.allclose(rr, torch.tensor([-1.5])), "reward = beta * (policy log-prob - reference log-prob)"
assert abs(loss.item() - 0.126928) < 1e-5, "loss = -log sigmoid(chosen reward - rejected reward), averaged"
big = dpo_loss(torch.tensor([0.0]), torch.tensor([-2000.0]), torch.tensor([0.0]), torch.tensor([0.0]), 1.0)[0]
assert torch.isfinite(big), "Use F.logsigmoid: log(sigmoid(x)) turns into log(0) for large gaps"
print(f"Untrained policy on {len(same)} pairs: loss {dpo_loss(same, same, same, same, 1.0)[0].item():.4f}   log 2 = {math.log(2):.4f}")

## Reward accuracy and margin

In [ ]:
def reward_metrics(chosen_rewards, rejected_rewards):
    margin=torch.mean(chosen_rewards-rejected_rewards)
    accuracy=(chosen_rewards>rejected_rewards).float()
    accuracy=torch.mean(accuracy)
    return {"margin":margin.item(),"accuracy":accuracy.item()}

m = reward_metrics(torch.tensor([1.0, 0.0, 2.0, 0.5]), torch.tensor([0.0, 0.0, 3.0, -0.5]))
assert m is not None and abs(m["accuracy"] - 0.5) < 1e-6, "A tie does not count: chosen must be strictly higher"
assert abs(m["margin"] - 0.25) < 1e-6, "margin = average of chosen reward minus rejected reward"
print(m)

## Connect the pieces

dpo_step connects your functions. It scores the chosen and rejected replies with the policy (TODO 1), computes the loss and rewards against the stored reference scores (TODO 3), and logs the metrics (TODO 4) with the average log-probabilities. On the untrained policy the loss is log 2 = 0.6931, the two log-probabilities are about -1.8, and accuracy and margin are 0 because every reward is exactly 0, which counts as a tie.

In [ ]:
def dpo_step(policy, batch, beta):
    policy_chosen = sequence_logprob(policy(batch["chosen"]), batch["chosen"], batch["chosen_mask"])
    policy_rejected = sequence_logprob(policy(batch["rejected"]), batch["rejected"], batch["rejected_mask"])
    loss, chosen_rewards, rejected_rewards = dpo_loss(
        policy_chosen, policy_rejected, batch["ref_chosen"], batch["ref_rejected"], beta)
    stats = {
        "chosen_logp": policy_chosen.mean().item(),
        "rejected_logp": policy_rejected.mean().item(),
        **reward_metrics(chosen_rewards, rejected_rewards),
    }
    return loss, stats


first_batch = {key: value[:16] for key, value in data.items()}
loss, stats = dpo_step(policy, first_batch, beta=1.0)
print(f"loss {loss.item():.4f}")
for key, value in stats.items():
    print(f"{key:14s} {value:8.3f}")

## Train with DPO

train_dpo runs 100 steps on random batches of 16 pairs, using gradient descent with momentum and a step size of 0.005 / beta. (Large models usually train DPO with Adam, but at this tiny scale Adam rescales the near-zero gradients of words the model is already sure of and breaks it, so plain momentum is used here. This is a choice for this toy, not a recommendation.) The DPO gradient is beta times sigmoid(rejected reward - chosen reward) times the log-probability gradients, so it shrinks with beta; dividing the step size by beta cancels that factor and gives every beta you will try the same first steps. Adam does something similar on its own, because it divides out the overall size of the gradient. Each line averages the last 20 steps. The loss falls from log 2 to about 0, accuracy reaches 1.00, and the margin (in reward units, beta times the log-probability gap) grows to about 8. Notice the two log-probabilities. The rejected reply falls from about -1.8 to about -9.6, while the chosen reply only rises from about -1.8 to about -1.2. The cell takes about 2 seconds.

In [ ]:
beta = 1.0
history = train_dpo(policy, lambda model, batch: dpo_step(model, batch, beta), data, steps=100, lr=0.005 / beta)

## Did the tone change?

The trained policy answers the same 6 requests, now always politely. The table compares the base model and the policy on 120 sampled replies to held-out requests. Polite rises from about 0.54 to 1.00, rude falls to 0.00, and on task stays at 1.00. Perplexity on neutral sentences moves only from about 3.43 to 3.46, so the change was targeted.

In [ ]:
for prompt, reply in zip(prompts[:6], generate(policy, prompts[:6], seed=1)):
    print(f"{prompt:32s} -> {reply}")

after_stats, _ = evaluate_replies(policy)
print(f"\n{'':10s}{'before':>8s}{'after':>8s}")
for key in ["polite", "rude", "on_task"]:
    print(f"{key:10s}{base_stats[key]:8.2f}{after_stats[key]:8.2f}")
print(f"{'perplexity':10s}{base_perplexity:8.2f}{perplexity(policy, neutral):8.2f}   (neutral sentences)")

## Where did the margin come from?

mean_logps averages the log-probability of all 600 chosen and rejected replies. The rude replies fell by almost 8 (from about -1.8 to -9.6, so they became thousands of times less likely), while the polite ones rose by only about 0.6. The token tables show where the change happened. Every shared word stays at about 0, which means certain, and the whole change sits on the first word of the ending. The rude "you" (of "you are so annoying") fell from -1.66 to -9.33. The polite "thank" rose only from -1.79 to -1.65, because the freed probability is shared among all three polite endings. DPO cares only about the gap between chosen and rejected, and lowering the rejected reply is the easiest way to widen it.

In [ ]:
def mean_logps(model):
    with torch.no_grad():
        chosen = sequence_logprob(model(data["chosen"]), data["chosen"], data["chosen_mask"])
        rejected = sequence_logprob(model(data["rejected"]), data["rejected"], data["rejected_mask"])
    return chosen.mean().item(), rejected.mean().item()


before, after = mean_logps(reference), mean_logps(policy)
print(f"average log-prob over all {len(pairs)} pairs   before   after")
print(f"  chosen (polite)                     {before[0]:7.2f} {after[0]:7.2f}")
print(f"  rejected (rude)                     {before[1]:7.2f} {after[1]:7.2f}\n")
show_token_logprobs(reference, policy, data["chosen"][0], data["chosen_mask"][0], "chosen")
print()
show_token_logprobs(reference, policy, data["rejected"][0], data["rejected_mask"][0], "rejected")

## Measure drift on neutral text

In [ ]:
def token_kl(policy_logits, ref_logits, mask):
    policy_soft=F.softmax(policy_logits,dim=-1)
    policy_logsoft=F.log_softmax(policy_logits,dim=-1)
    ref_logsoft=F.log_softmax(ref_logits,dim=-1)
    Kl_token=torch.sum(policy_soft*(policy_logsoft-ref_logsoft),dim=-1)
    if(torch.sum(mask)==0):
     return torch.tensor(0.0)
    kl=torch.sum(mask*Kl_token)/torch.sum(mask)
    return kl

p_toy = torch.tensor([[[0.0, 0.0]]])
q_toy = torch.tensor([[[math.log(3.0), 0.0]]])
assert token_kl(p_toy, p_toy, torch.ones(1, 1)).item() == 0.0, "A model has zero KL to itself"
assert abs(token_kl(p_toy, q_toy, torch.ones(1, 1)).item() - (0.5 * math.log(0.5 / 0.75) + 0.5 * math.log(0.5 / 0.25))) < 1e-6, "Sum p * (log p - log q) over the vocabulary"
assert abs(token_kl(torch.cat([p_toy, p_toy], 1), torch.cat([q_toy, p_toy], 1), torch.tensor([[1.0, 0.0]])).item() - token_kl(p_toy, q_toy, torch.ones(1, 1)).item()) < 1e-6, "Average only over positions where mask is 1"

neutral_tokens, neutral_mask = encode_sentences(neutral)


def drift(model):
    with torch.no_grad():
        return token_kl(model(neutral_tokens)[:, :-1], reference(neutral_tokens)[:, :-1], neutral_mask[:, 1:].float()).item()


print(f"KL per token on neutral sentences, reference vs itself: {drift(reference):.4f}")
print(f"KL per token on neutral sentences, policy (beta 1):     {drift(policy):.4f}")

## Turn beta down

Now train two fresh copies of the base model with smaller beta values, 0.1 and 0.03, with everything else the same. Keep two settings apart. Beta decides how big a log-probability gap the loss asks for. Each pair stops pushing once beta times its gap is several units, so beta 0.1 asks for a gap 10 times larger than beta 1. The step size decides how fast a run moves toward that gap, and it is 0.005 / beta here, so every run starts at the same speed and only the stopping point differs. With one fixed step size for every beta the picture changes, because a small beta then also takes tiny steps, and after 100 steps it would have moved the least (we measured this). So the drift you see next comes from beta asking for a bigger gap and the run having the steps to reach for it. With beta 0.1 the rejected log-prob falls to about -42 and the sample replies still look fine. With beta 0.03 the log (each line averages 50 steps) shows the rejected log-prob plunging to about -159 and the chosen one falling too, to about -13, and the replies turn into word salad mixing polite and rude words.

In [ ]:
results = {1.0: (after_stats, perplexity(policy, neutral), drift(policy), mean_logps(policy))}
for small_beta in [0.1, 0.03]:
    model = copy.deepcopy(base)
    print(f"beta {small_beta}:")
    log = train_dpo(model, lambda m, batch: dpo_step(m, batch, small_beta), data, steps=100, lr=0.005 / small_beta, log_every=50)
    results[small_beta] = (evaluate_replies(model)[0], perplexity(model, neutral), drift(model), mean_logps(model))
    for prompt, reply in zip(prompts[6:9], generate(model, prompts[6:9], seed=1)):
        print(f"  {prompt:32s} -> {reply}")
    print()

## Turn beta up, then compare

Train one more copy with a large beta, 10, then compare every run. In the table, polite and on task are shares of 120 sampled replies, perplexity and KL are measured on 200 neutral sentences (KL is TODO 5, 0 means unchanged), and chosen lp and rejected lp are average log-probabilities over the 600 pairs. Beta 10 moved the least, with KL about 0.001, the rejected log-prob only falling to about -4, and 0.92 of replies polite. Beta 1 reached 1.00 polite with KL about 0.014. Beta 0.1 is also 1.00 polite, so its preference numbers look perfect, but its KL is about 16 times larger (0.22) and its neutral perplexity rose from 3.4 to about 6, so it drifted far for nothing. Beta 0.03 broke the model, with perplexity over 1000 and only half the replies on task. In that row both log-probs fell, the chosen one from -1.8 to about -8.6, yet the gap between the two columns is the largest of all (about 210), because the rejected one fell about 30 times further (to about -219). The DPO loss only sees the difference, so it calls this a success. In this toy the chosen log-prob falls only in the broken run; in large models both falling is common even in healthy runs. Keep the scale in mind. The tone here is one word in a 95-word model trained with large steps, so the effects are much larger and faster than in real models, where beta 0.1 is a usual, safe default. The lesson is the trade-off, not the numbers.

In [ ]:
model = copy.deepcopy(base)
log = train_dpo(model, lambda m, batch: dpo_step(m, batch, 10.0), data, steps=100, lr=0.005 / 10.0, log_every=50)
results[10.0] = (evaluate_replies(model)[0], perplexity(model, neutral), drift(model), mean_logps(model))

print(f"\n{'beta':>6s}{'polite':>8s}{'on task':>9s}{'perplexity':>12s}{'KL':>8s}{'chosen lp':>11s}{'rejected lp':>13s}")
print(f"{'base':>6s}{base_stats['polite']:8.2f}{base_stats['on_task']:9.2f}{base_perplexity:12.2f}{0:8.3f}"
      f"{before[0]:11.2f}{before[1]:13.2f}")
for b in sorted(results):
    stats, ppl, kl, (chosen_lp, rejected_lp) = results[b]
    print(f"{b:6g}{stats['polite']:8.2f}{stats['on_task']:9.2f}{ppl:12.2f}{kl:8.3f}{chosen_lp:11.2f}{rejected_lp:13.2f}")

## Optional: IPO aims for a fixed gap

In [ ]:
def ipo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, tau):
    h=policy_chosen-ref_chosen-policy_rejected+ref_rejected
    h=h-1/(2*tau)
    loss=torch.dot(h.T,h)/h.shape[0]
    return loss

assert ipo_loss(torch.tensor([5.0]), torch.tensor([0.0]), torch.tensor([0.0]), torch.tensor([0.0]), 0.1).item() == 0.0, "A gap of exactly 1 / (2 tau) gives zero loss"
assert abs(ipo_loss(torch.tensor([0.0, 9.0]), torch.tensor([0.0, 0.0]), torch.zeros(2), torch.zeros(2), 0.1).item() - 20.5) < 1e-5, "Square the distance from the target, then average"


def ipo_step(policy, batch, tau):
    policy_chosen = sequence_logprob(policy(batch["chosen"]), batch["chosen"], batch["chosen_mask"])
    policy_rejected = sequence_logprob(policy(batch["rejected"]), batch["rejected"], batch["rejected_mask"])
    loss = ipo_loss(policy_chosen, policy_rejected, batch["ref_chosen"], batch["ref_rejected"], tau)
    return loss, {"chosen_logp": policy_chosen.mean().item(), "rejected_logp": policy_rejected.mean().item()}


tau = 0.03
model = copy.deepcopy(base)
log = train_dpo(model, lambda m, batch: ipo_step(m, batch, tau), data, steps=100, lr=0.0025 * tau, log_every=50)
ipo_stats = evaluate_replies(model)[0]
print(f"\n{'':14s}{'polite':>8s}{'on task':>9s}{'perplexity':>12s}{'KL':>8s}")
s, ppl, kl, _ = results[0.03]
print(f"{'DPO beta 0.03':14s}{s['polite']:8.2f}{s['on_task']:9.2f}{ppl:12.2f}{kl:8.3f}")
print(f"{'IPO tau 0.03':14s}{ipo_stats['polite']:8.2f}{ipo_stats['on_task']:9.2f}{perplexity(model, neutral):12.2f}{drift(model):8.3f}")